# 5. Result Analysis

**Purpose:** Analyse the bias-corrected per-grid CSVs (from `4a` or `4b`) at the **basin-average** scale and produce:

1. **Annual mean** and **annual max** rainfall time series per GCM, with the multi-model mean.
2. **Mann–Kendall** trend test (table + summary).
3. **Frequency analysis of the annual maximum daily rainfall, per GCM**, in the manner of 水文統計ユーティリティ (国総研): candidate distributions fitted to each model's annual-maximum series, goodness of fit by **SLSC** (accepted when ≤ 0.04), selection by the smallest **Jackknife estimation error** of the 100-year value, and return-period rainfall tables. The multi-model result is the **mean / median / range of the per-model return-period values**, plus a **pooled** fit (all models' annual maxima as one sample) for reference.

All outputs (figures + tables) are saved to `OUTPUT_DIR/analysis/`.

## Why per model

Averaging the ten models' daily series before taking annual maxima (the previous "Ensemble" series) destroys the extremes, because each model has its extreme events on different days: the annual maximum of the averaged series is ~30 mm where each model's own annual maximum is ~130–150 mm, and the fitted 100-year value drops from ~280 mm to ~55 mm. Frequency analysis is therefore done on each model's own annual-maximum series, and the models are combined at the level of the return-period values.

## Inputs (read from Drive)

- `OUTPUT_DIR/his_corr/his_id_*_corrected.csv` — bias-corrected historical (one column per GCM, one row per day)
- `OUTPUT_DIR/{ssp}_corr/fut_{ssp}_id_*_corrected.csv` — bias-corrected future scenarios
- `OUTPUT_DIR/gcm_grid_shp/gcm_grid_cells.csv` — cell weights (`frac_basin`, written by `2_DataDownload.ipynb`); equal weights if absent
- `OUTPUT_DIR/his_GSMaP_orig/` or `OUTPUT_DIR/his_obs_orig/` — the observation **that was used for the correction** (auto-detected by comparing per-cell maxima with the corrected historical; override with `OBS_SOURCE`)

## Outputs (`OUTPUT_DIR/analysis/`)

- `basin_avg_{scenario}.csv`, `basin_avg_Observation.csv` — basin-averaged daily series (date × model)
- `annual_mean_all.png/.pdf`, `annual_max_all.png/.pdf` (stacked panels, same colour scheme), `mk_trends.csv`
- `prob_plot_{scenario}.png/.pdf` and `prob_plot_all.png/.pdf` — Gumbel probability paper per scenario and all scenarios stacked (grey: individual GCMs, black dashed: multi-model mean of the fitted quantiles, black: pooled-sample fit, red: observation); figures are 300 dpi PNG plus vector PDF with editable text
- `return_periods_by_model.csv` — selected distribution, SLSC, Jackknife error and return-period rainfall for every (scenario, model), the pooled sample and the observation
- `return_periods_summary.csv` — per scenario: mean / median / min / max over models, pooled, observation
- `fit_scores.csv` — SLSC and Jackknife error of every candidate distribution for every series
- `return_periods.xlsx` — all of the above in one workbook

## Update log

- 2026-09-12 Kurihara
  1. Frequency analysis per GCM (annual maxima of each model's basin-average series) instead of the annual maxima of the multi-model daily mean; multi-model statistics taken on the return-period values; pooled fit added.
  2. Estimation in the manner of 水文統計ユーティリティ: Cunnane plotting position (a = 0.4); SLSC on each distribution's own standardized variate with the |s(0.99) − s(0.01)| denominator; Jackknife estimate and standard error of the 100-year value; acceptance SLSC ≤ 0.04; selection by the smallest Jackknife error. L-moments for Exp / Gumbel / GEV / LN2; MLE for SqrtEt; quantile method (LN3Q) and product moments (LN3PM) for the 3-parameter lognormal; log-space (LogP3) and real-space (LP3Rs) moments for log-Pearson III; Iwai's method. The Ishihara–Takase method is not included (its lower-bound estimator could not be reproduced reliably); the previous `IshiTaka`, `LN3Q`, `LN3PM` and `Iwai` entries were all plain 2-parameter fits.
  3. Observation source auto-detected from the corrected historical maxima (previously GSMaP was always preferred, even when the correction used gauges).
  4. Basin average weighted by each cell's share of the basin (`frac_basin`) when available.
  5. "Ensemble" in the annual series = mean over models of the per-model annual values.
  6. Environment cell runs outside Colab (`OUTPUT_DIR` from the environment variable or the default path).
  7. Figures in publication style: individual GCMs in grey, multi-model mean and pooled fit in black, observation in red; scenarios stacked vertically with a common y-axis and panel letters; 300 dpi PNG plus PDF. A scenario whose corrected files hold fewer than 6 years (e.g. from a QUICK_TEST download) is reported and skipped instead of stopping the notebook.


In [ ]:
# ============================================================
# Environment Setup (Colab / Local)
# OUTPUT_DIR must be the SAME folder used by 4a / 4b.
# ============================================================
import os
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _mount_drive():
    if not IN_COLAB:
        print('Not running in Colab: Drive mount skipped.')
        return
    if os.path.ismount('/content/drive'):
        return
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
    try:
        drive.mount('/content/drive')
    except ValueError as e:
        print(f'First mount attempt failed: {e}')
        print('  Retrying with force_remount=True ...')
        drive.mount('/content/drive', force_remount=True)


_mount_drive()

_default_out = '/content/drive/MyDrive/Downscaling/Output' if IN_COLAB else os.path.join(os.getcwd(), 'Output')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', _default_out)
if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(f'OUTPUT_DIR not found: {OUTPUT_DIR}. Point it at the Output folder used by 4a / 4b '
                            f'(this notebook does not create one).')

ANALYSIS_DIR = f'{OUTPUT_DIR}/analysis'
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f'OUTPUT_DIR  : {OUTPUT_DIR}')
print(f'ANALYSIS_DIR: {ANALYSIS_DIR}')


In [ ]:
# ============================================================
# Imports
# ============================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats, optimize, special
import warnings
warnings.filterwarnings('ignore')

try:
    import pymannkendall as mk
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'pymannkendall'], check=True)
    import pymannkendall as mk

from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

# Publication-style defaults: 9 pt sans-serif, thin axes, light grid, editable text in PDF
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 9.5, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 7.5, 'legend.frameon': False,
    'axes.linewidth': 0.7, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
COL_GCM, COL_GCM_PT, COL_ENS, COL_OBS = '0.55', '0.72', 'black', '#c0392b'   # grey GCM lines / light-grey points / black ensemble / red observation


def save_figure(fig, base):
    """PNG (300 dpi) and PDF (vector) with the same base name."""
    for ext in ('png', 'pdf'):
        fig.savefig(f'{base}.{ext}')
    print(f'  Saved: {base}.png / .pdf')


## Section 1: Load corrected CSVs and compute the basin average

For each scenario (`historical`, `ssp126`, `ssp245`, `ssp370`, `ssp585`):
1. Read all per-grid corrected CSVs.
2. Average across grid cells → basin-average daily rainfall per GCM model. The average is **weighted by each cell's share of the basin** (`frac_basin` in `gcm_grid_shp/gcm_grid_cells.csv`, written by `2_DataDownload.ipynb`); if that file is missing, cells are weighted equally.
3. Save to `analysis/basin_avg_{scenario}.csv` (column = model, row = date).

The observation is loaded the same way (same cells, same weights). Which observation was used for the correction is detected automatically: the corrected historical series has, per cell, exactly the maximum of the observation it was matched to, so the source whose per-cell maxima agree is chosen (`OBS_SOURCE = 'auto'`; set `'GSMaP'` or `'Gauge'` to force one).

If a scenario folder is missing it is skipped.


In [ ]:
# ============================================================
# Section 1: Basin-average daily rainfall per scenario
# ============================================================
OBS_SOURCE = 'auto'   # 'auto' | 'GSMaP' | 'Gauge'

SCENARIOS = [
    ('historical', f'{OUTPUT_DIR}/his_corr', r'^his_id_(\d+)_corrected\.csv$'),
    ('ssp126',     f'{OUTPUT_DIR}/ssp126_corr', r'^fut_ssp126_id_(\d+)_corrected\.csv$'),
    ('ssp245',     f'{OUTPUT_DIR}/ssp245_corr', r'^fut_ssp245_id_(\d+)_corrected\.csv$'),
    ('ssp370',     f'{OUTPUT_DIR}/ssp370_corr', r'^fut_ssp370_id_(\d+)_corrected\.csv$'),
    ('ssp585',     f'{OUTPUT_DIR}/ssp585_corr', r'^fut_ssp585_id_(\d+)_corrected\.csv$'),
]
OBS_SOURCES = {
    'GSMaP': (f'{OUTPUT_DIR}/his_GSMaP_orig', r'^his_GSMaP_id_(\d+)\.csv$', 'GSMaP'),
    'Gauge': (f'{OUTPUT_DIR}/his_obs_orig',   r'^his_obs_id_(\d+)\.csv$',   'Rain'),
}


def files_by_id(folder, regex):
    rx = re.compile(regex)
    out = {}
    if os.path.isdir(folder):
        for p in Path(folder).glob('*.csv'):
            m = rx.match(p.name)
            if m:
                out[int(m.group(1))] = p
    return out


def read_daily(path, value_col=None):
    """Per-cell CSV -> DataFrame indexed by date (all models, or the single observation column)."""
    df = pd.read_csv(path)
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if value_col is not None:
        df = df[[value_col]]
    return df.apply(pd.to_numeric, errors='coerce').fillna(0)


# ---- cell weights: share of the basin per cell ----
cells_csv = f'{OUTPUT_DIR}/gcm_grid_shp/gcm_grid_cells.csv'
cell_weight = None
if os.path.exists(cells_csv):
    _cells = pd.read_csv(cells_csv)
    if {'id', 'frac_basin'} <= set(_cells.columns):
        cell_weight = dict(zip(_cells['id'].astype(int), _cells['frac_basin'].astype(float)))
        print(f'Cell weights (frac_basin) from {cells_csv}: ' + ', '.join(f'id {i}: {w:.3f}' for i, w in sorted(cell_weight.items())))
if cell_weight is None:
    print('No gcm_grid_cells.csv with frac_basin found — cells are weighted equally.')


def weighted_basin_average(per_cell):
    """per_cell: {id: DataFrame(date x columns)} -> DataFrame, weighted by frac_basin (or equal)."""
    ids = sorted(per_cell)
    w = np.array([cell_weight.get(i, np.nan) if cell_weight else 1.0 for i in ids], dtype=float)
    if np.isnan(w).any() or w.sum() <= 0:
        w = np.ones(len(ids))
    w = w / w.sum()
    common = per_cell[ids[0]].columns
    idx = per_cell[ids[0]].index
    for i in ids[1:]:
        common = common.intersection(per_cell[i].columns)
        idx = idx.intersection(per_cell[i].index)
    acc = sum(per_cell[i].loc[idx, common] * wi for i, wi in zip(ids, w))
    return acc.sort_index(), dict(zip(ids, w))


basin_data, per_cell_max = {}, {}
for sc, folder, regex in SCENARIOS:
    fmap = files_by_id(folder, regex)
    if not fmap:
        print(f'  ⚠ {sc}: no files found, skipped')
        continue
    per_cell = {i: read_daily(p) for i, p in fmap.items()}
    per_cell_max[sc] = {i: float(df.max().max()) for i, df in per_cell.items()}
    df, w = weighted_basin_average(per_cell)
    basin_data[sc] = df
    df.to_csv(f'{ANALYSIS_DIR}/basin_avg_{sc}.csv')
    n_years = df.index.year.nunique()
    print(f'  ✓ {sc}: {df.shape[0]} days ({df.index.min().date()}..{df.index.max().date()}, {n_years} yr) × {df.shape[1]} models, '
          f'cells {sorted(fmap)} (weights {", ".join(f"{v:.2f}" for v in w.values())})'
          + ('  ⚠ fewer than 6 years: frequency analysis will skip this scenario' if n_years < 6 else ''))

print(f'\nLoaded scenarios: {list(basin_data)}')

# ---- Observation: the one the correction was matched to ----
obs_daily, obs_source_label = None, None
candidates = {}
for src, (folder, regex, col) in OBS_SOURCES.items():
    fmap = files_by_id(folder, regex)
    if fmap:
        candidates[src] = {i: read_daily(p, col).rename(columns={col: 'Observation'}) for i, p in fmap.items()}
if OBS_SOURCE != 'auto':
    if OBS_SOURCE not in candidates:
        raise FileNotFoundError(f'OBS_SOURCE = {OBS_SOURCE!r} but no files found for it.')
    obs_source_label = OBS_SOURCE
elif candidates:
    his_max = per_cell_max.get('historical', {})
    match = {}
    for src, pc in candidates.items():
        ids = [i for i in pc if i in his_max]
        if ids:
            match[src] = float(np.mean([abs(pc[i]['Observation'].max() - his_max[i]) for i in ids]))
    if match:
        obs_source_label = min(match, key=match.get)
        print('Observation source check (mean |obs max - corrected historical max| per cell, mm): '
              + ', '.join(f'{k}: {v:.1f}' for k, v in match.items()) + f' -> using {obs_source_label}')
        if match[obs_source_label] > 1.0:
            print('  ⚠ no observation source reproduces the corrected historical maxima — check that 4a/4b ran with these files')
    else:
        obs_source_label = next(iter(candidates))
if obs_source_label:
    obs_daily, _ = weighted_basin_average(candidates[obs_source_label])
    obs_daily.to_csv(f'{ANALYSIS_DIR}/basin_avg_Observation.csv')
    print(f'  ✓ Observation ({obs_source_label}): {len(obs_daily)} days, '
          f'{obs_daily.index.min().date()}..{obs_daily.index.max().date()}')
else:
    print('  ⚠ No observation found (neither his_GSMaP_orig nor his_obs_orig). Observation analysis will be skipped.')


## Section 2: Annual mean / max + Mann–Kendall trend

- **Annual mean** = mean of the daily basin-average rainfall in each calendar year, per model.
- **Annual max** = maximum daily basin-average rainfall in each calendar year, per model.
- **Ensemble** = mean over the models of the per-model annual values (for the annual max this is *not* the annual max of the averaged daily series).
- The Mann–Kendall trend test (`pymannkendall`) is run for each (scenario, model), the Ensemble and the Observation.


In [ ]:
# ============================================================
# Section 2a: Annual-aggregated series (GCM scenarios + Observation)
# ============================================================
annual_mean, annual_max = {}, {}
for sc, df in basin_data.items():
    am = df.groupby(df.index.year).mean()
    ax_ = df.groupby(df.index.year).max()
    am['Ensemble'] = am.mean(axis=1)
    ax_['Ensemble'] = ax_.mean(axis=1)      # mean of the per-model annual maxima
    annual_mean[sc], annual_max[sc] = am, ax_
    print(f'  {sc}: years {am.index.min()}–{am.index.max()}, models {[c for c in am.columns if c != "Ensemble"]}')

obs_annual_mean, obs_annual_max = {}, {}
if obs_daily is not None:
    obs_annual_mean['Observation'] = obs_daily.groupby(obs_daily.index.year).mean()
    obs_annual_max['Observation'] = obs_daily.groupby(obs_daily.index.year).max()
    print(f'  Observation ({obs_source_label}): years {obs_annual_mean["Observation"].index.min()}–{obs_annual_mean["Observation"].index.max()}')


In [ ]:
# ============================================================
# Section 2b: Plot annual mean / annual max per scenario
# (Observation overlay on the historical panel only)
# ============================================================
def _plot_annual(annual_dict, obs_annual_dict, ylabel, file_prefix):
    """Stacked panels (one per scenario): grey = individual GCMs, black = multi-model mean, red dashed = observation."""
    panels = [(sc, ann) for sc, ann in annual_dict.items() if len(ann) >= 2]
    for sc, ann in annual_dict.items():
        if len(ann) < 2:
            print(f'  {file_prefix}: {sc} has only {len(ann)} year(s) — left out of the figure')
    if not panels and obs_annual_dict:
        panels = [('historical', None)]
    if not panels:
        print(f'  {file_prefix}: no data, skipped')
        return
    # common y-axis over all panels (GCMs and observation)
    vals = [ann.to_numpy().ravel() for _, ann in panels if ann is not None] + [o.to_numpy().ravel() for o in obs_annual_dict.values()]
    allv = np.concatenate(vals) if vals else np.array([0, 1])
    ylim = (0 if allv.min() >= 0 else allv.min() * 1.05, allv.max() * 1.08)
    n = len(panels)
    fig, axes = plt.subplots(n, 1, figsize=(6.5, 2.3 * n + 0.4), squeeze=False)
    for i, (ax, (sc, ann)) in enumerate(zip(axes[:, 0], panels)):
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_ylim(*ylim)
        if ann is not None:
            for col in [c for c in ann.columns if c != 'Ensemble']:
                ax.plot(ann.index, ann[col].values, color=COL_GCM, linewidth=0.6, alpha=0.9)
            ax.plot(ann.index, ann['Ensemble'].values, color=COL_ENS, linewidth=1.8)
        if sc == 'historical':
            for label, obs_ann in obs_annual_dict.items():
                ax.plot(obs_ann.index, obs_ann.iloc[:, 0].values, color=COL_OBS, linewidth=1.5, linestyle='--')
        ax.text(0.01, 0.95, f'({chr(97 + i)}) {sc}', transform=ax.transAxes, va='top', ha='left', fontweight='bold')
        ax.set_ylabel(ylabel)
        ax.margins(x=0.01)
    axes[-1, 0].set_xlabel('Year')
    handles = [Line2D([], [], color=COL_GCM, lw=0.8, label='Individual GCMs'),
               Line2D([], [], color=COL_ENS, lw=1.8, label='Multi-model mean')]
    if obs_annual_dict:
        handles.append(Line2D([], [], color=COL_OBS, lw=1.5, ls='--', label=f'Observation ({obs_source_label})'))
    axes[0, 0].legend(handles=handles, loc='upper right', ncol=len(handles))
    fig.tight_layout(h_pad=0.6)
    save_figure(fig, f'{ANALYSIS_DIR}/{file_prefix}_all')
    plt.show()


_plot_annual(annual_mean, obs_annual_mean, 'Annual mean (mm/day)', 'annual_mean')
_plot_annual(annual_max, obs_annual_max, 'Annual max. daily (mm)', 'annual_max')


In [ ]:
# ============================================================
# Section 2c: Mann–Kendall trend table
# ============================================================
def _mk_row(series, scenario, metric, model):
    s = series.dropna()
    if len(s) < 4:
        return None
    r = mk.original_test(s.values)
    return {'scenario': scenario, 'metric': metric, 'model': model, 'trend': r.trend,
            'p_value': round(r.p, 4), 'z': round(r.z, 3), 'slope_per_year': round(r.slope, 4)}


rows = []
for metric_name, src in (('annual_mean', annual_mean), ('annual_max', annual_max)):
    for sc, ann in src.items():
        for col in ann.columns:
            row = _mk_row(ann[col], sc, metric_name, col)
            if row is not None:
                rows.append(row)
for metric_name, src in (('annual_mean', obs_annual_mean), ('annual_max', obs_annual_max)):
    for label, ann in src.items():
        row = _mk_row(ann.iloc[:, 0], 'historical', metric_name, f'Observation ({obs_source_label})')
        if row is not None:
            rows.append(row)

mk_df = pd.DataFrame(rows)
mk_csv = f'{ANALYSIS_DIR}/mk_trends.csv'
mk_df.to_csv(mk_csv, index=False)
print(f'Mann–Kendall results: {mk_csv}')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
summary_view = mk_df[mk_df['model'].str.startswith(('Ensemble', 'Observation'))].sort_values(['metric', 'scenario', 'model'])
print('\nEnsemble + Observation trend summary:')
print(summary_view[['metric', 'scenario', 'model', 'trend', 'p_value', 'z', 'slope_per_year']].to_string(index=False))


## Section 3: Frequency analysis of the annual maximum daily rainfall (per GCM)

Procedure, following 水文統計ユーティリティ as far as it can be reproduced here:

1. **Sample**: the annual maximum of the basin-average daily rainfall, one series per (scenario, GCM); the observation is one more series. A **pooled** sample (all models' annual maxima of a scenario together) is analysed as well.
2. **Candidate distributions and parameter estimation**
   | key | distribution | estimation |
   |---|---|---|
   | `Exp` | exponential (2 parameters) | L-moments |
   | `Gumbel` | Gumbel | L-moments |
   | `SqrtEt` | square-root exponential-type maximum (Etoh–Murota) | maximum likelihood |
   | `Gev` | generalised extreme value | L-moments (Hosking) |
   | `LN2LM` | 2-parameter lognormal | L-moments of log x |
   | `LN3Q` | 3-parameter lognormal | lower bound from the 5 / 50 / 95 % quantiles, then moments of log(x − a) |
   | `LN3PM` | 3-parameter lognormal | product moments (mean, variance, skew) |
   | `LogP3` | log-Pearson III | moments of log x |
   | `LP3Rs` | log-Pearson III | moments in real space (mean, variance, skew of x) |
   | `Iwai` | Iwai's method (lognormal with lower bound −b) | b from the m = N/10 largest–smallest pairs and the geometric mean |

   The Ishihara–Takase method of the utility is not included.
3. **Plotting position**: Cunnane, p = (i − 0.4)/(N + 0.2).
4. **SLSC** = √(Σ(s_i − s*_i)²/N) / |s(0.99) − s(0.01)|, where s is the standardized variate of the distribution (Gumbel reduced variate for Gumbel / GEV / SqrtEt, standard normal for the lognormal family, standardized exponential for Exp, Pearson III frequency factor for the log-Pearson family); s_i is the standardized value of the i-th observation under the fitted distribution and s*_i the one at its plotting position. A fit is **accepted when SLSC ≤ 0.04**.
5. **Jackknife**: leave-one-out refits give the bias-corrected estimate and the standard error of the 100-year value (`JK_T`). Among the accepted distributions the one with the **smallest Jackknife standard error** is selected; if none is accepted the smallest SLSC is taken and flagged.
6. **Return-period rainfall** of the selected distribution for T = 2 … 400 years, per series. The multi-model result is the **mean, median, min and max over the models** of these values; the pooled fit is reported next to it.


In [ ]:
# ============================================================
# Section 3a: Frequency-analysis library
# ============================================================
SLSC_THRESHOLD = 0.04
JK_T = 100                                             # return period whose Jackknife error selects the distribution
ALL_RPS = [2, 5, 10, 20, 30, 50, 80, 100, 150, 200, 400]
EULER = 0.5772156649015329
_EPS = 1e-9


def cunnane(n, a=0.4):
    i = np.arange(1, n + 1)
    return (i - a) / (n + 1 - 2 * a)


def gumbel_y(p):
    return -np.log(-np.log(np.asarray(p, dtype=float)))


def sample_lmoments(x):
    """Unbiased sample L-moments: returns (l1, l2, t3, t4)."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    i = np.arange(1, n + 1, dtype=float)
    b0 = x.mean()
    b1 = np.sum((i - 1) / (n - 1) * x) / n
    b2 = np.sum((i - 1) * (i - 2) / ((n - 1) * (n - 2)) * x) / n
    b3 = np.sum((i - 1) * (i - 2) * (i - 3) / ((n - 1) * (n - 2) * (n - 3)) * x) / n
    l1, l2 = b0, 2 * b1 - b0
    l3, l4 = 6 * b2 - 6 * b1 + b0, 20 * b3 - 30 * b2 + 12 * b1 - b0
    return l1, l2, (l3 / l2 if l2 > 0 else 0.0), (l4 / l2 if l2 > 0 else 0.0)


def pearson3_K(p, g):
    """Frequency factor of the Pearson III distribution with skew g (standardized variate)."""
    p = np.asarray(p, dtype=float)
    if abs(g) < 1e-6:
        return stats.norm.ppf(p)
    a = 4.0 / g ** 2
    if g > 0:
        return (stats.gamma.ppf(p, a) - a) / np.sqrt(a)
    return -(stats.gamma.ppf(1 - p, a) - a) / np.sqrt(a)


def pearson3_cdf_std(z, g):
    """CDF of the standardized Pearson III variate z with skew g."""
    z = np.asarray(z, dtype=float)
    if abs(g) < 1e-6:
        return stats.norm.cdf(z)
    a = 4.0 / g ** 2
    if g > 0:
        return stats.gamma.cdf(z * np.sqrt(a) + a, a)
    return 1 - stats.gamma.cdf(-z * np.sqrt(a) + a, a)


class Dist:
    name = ''
    kind = 'gumbel'          # standardized variate: 'gumbel' | 'normal' | 'exp' | 'pearson'

    def __init__(self, **params):
        self.p = params

    @classmethod
    def fit(cls, x):
        raise NotImplementedError

    def quantile(self, p):
        raise NotImplementedError

    def cdf(self, x):
        raise NotImplementedError

    def reduced(self, p):
        p = np.asarray(p, dtype=float)
        if self.kind == 'gumbel':
            return gumbel_y(p)
        if self.kind == 'normal':
            return stats.norm.ppf(p)
        if self.kind == 'exp':
            return -np.log(1 - p)
        return pearson3_K(p, self.p['g'])

    def reduced_of_x(self, x):
        return self.reduced(np.clip(self.cdf(np.asarray(x, dtype=float)), _EPS, 1 - _EPS))


class Exp(Dist):
    name, kind = 'Exp', 'exp'

    @classmethod
    def fit(cls, x):
        l1, l2, _, _ = sample_lmoments(x)
        b = 2 * l2
        return cls(a=l1 - b, b=b) if b > 0 else None

    def quantile(self, p):
        return self.p['a'] - self.p['b'] * np.log(1 - np.asarray(p, dtype=float))

    def cdf(self, x):
        return 1 - np.exp(-np.maximum(np.asarray(x, dtype=float) - self.p['a'], 0) / self.p['b'])


class Gumbel(Dist):
    name, kind = 'Gumbel', 'gumbel'

    @classmethod
    def fit(cls, x):
        l1, l2, _, _ = sample_lmoments(x)
        if l2 <= 0:
            return None
        beta = l2 / np.log(2)
        return cls(mu=l1 - EULER * beta, beta=beta)

    def quantile(self, p):
        return self.p['mu'] + self.p['beta'] * gumbel_y(p)

    def cdf(self, x):
        return np.exp(-np.exp(-(np.asarray(x, dtype=float) - self.p['mu']) / self.p['beta']))


class GEV(Dist):
    name, kind = 'Gev', 'gumbel'

    @classmethod
    def fit(cls, x):
        l1, l2, t3, _ = sample_lmoments(x)
        if l2 <= 0:
            return None
        c = 2 / (3 + t3) - np.log(2) / np.log(3)
        k = 7.8590 * c + 2.9554 * c ** 2
        if abs(k) < 1e-6:
            k = 1e-6
        g1 = special.gamma(1 + k)
        alpha = l2 * k / ((1 - 2 ** (-k)) * g1)
        xi = l1 - alpha * (1 - g1) / k
        return cls(xi=xi, alpha=alpha, k=k) if alpha > 0 else None

    def quantile(self, p):
        xi, a, k = self.p['xi'], self.p['alpha'], self.p['k']
        return xi + a * (1 - (-np.log(np.asarray(p, dtype=float))) ** k) / k

    def cdf(self, x):
        xi, a, k = self.p['xi'], self.p['alpha'], self.p['k']
        t = 1 - k * (np.asarray(x, dtype=float) - xi) / a
        with np.errstate(invalid='ignore', divide='ignore'):
            F = np.where(t > 0, np.exp(-np.power(np.maximum(t, 1e-300), 1 / k)), 1.0 if k > 0 else 0.0)
        return F


class LN2LM(Dist):
    name, kind = 'LN2LM', 'normal'

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=float)
        if (x <= 0).any():
            return None
        l1, l2, _, _ = sample_lmoments(np.log(x))
        return cls(a=0.0, mu=l1, sigma=np.sqrt(np.pi) * l2) if l2 > 0 else None

    def quantile(self, p):
        return self.p['a'] + np.exp(self.p['mu'] + self.p['sigma'] * stats.norm.ppf(np.asarray(p, dtype=float)))

    def cdf(self, x):
        x = np.asarray(x, dtype=float)
        with np.errstate(invalid='ignore', divide='ignore'):
            z = (np.log(np.maximum(x - self.p['a'], 1e-300)) - self.p['mu']) / self.p['sigma']
        return np.where(x > self.p['a'], stats.norm.cdf(z), 0.0)


class LN3Q(LN2LM):
    """3-parameter lognormal, lower bound from the 5 / 50 / 95 % quantiles (Cunnane positions)."""
    name = 'LN3Q'

    @classmethod
    def fit(cls, x):
        xs = np.sort(np.asarray(x, dtype=float))
        n = len(xs)
        if n < 6:
            return None
        pp = cunnane(n)
        x05, x50, x95 = np.interp([0.05, 0.5, 0.95], pp, xs)
        den = x95 + x05 - 2 * x50
        if den <= 0:
            return None
        a = (x95 * x05 - x50 ** 2) / den
        if a >= xs[0]:
            return None
        y = np.log(xs - a)
        return cls(a=a, mu=y.mean(), sigma=y.std(ddof=1))


class LN3PM(LN2LM):
    """3-parameter lognormal by product moments (mean, variance, skew of x)."""
    name = 'LN3PM'

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=float)
        m, s, g = x.mean(), x.std(ddof=1), stats.skew(x, bias=False)
        if g <= 0 or s <= 0:
            return None
        f = lambda w: (w + 2) * np.sqrt(w - 1) - g          # w = exp(sigma_y^2)
        try:
            w = optimize.brentq(f, 1 + 1e-12, 1e6)
        except ValueError:
            return None
        sigma = np.sqrt(np.log(w))
        emu = s / np.sqrt(w * (w - 1))
        a = m - emu * np.sqrt(w)
        if a >= x.min():
            return None
        return cls(a=a, mu=np.log(emu), sigma=sigma)


class Iwai(LN2LM):
    """Iwai's method: log(x + b) normal; b from the m = N/10 largest-smallest pairs and the geometric mean."""
    name = 'Iwai'

    @classmethod
    def fit(cls, x):
        xs = np.sort(np.asarray(x, dtype=float))[::-1]        # descending
        n = len(xs)
        if n < 6 or (xs <= 0).any():
            return None
        xg = np.exp(np.mean(np.log(xs)))
        m = max(1, int(round(n / 10)))
        bs = []
        for i in range(m):
            hi, lo = xs[i], xs[n - 1 - i]
            den = hi + lo - 2 * xg
            if abs(den) > 1e-12:
                bs.append((xg ** 2 - hi * lo) / den)
        if not bs:
            return None
        b = float(np.mean(bs))
        if (xs + b <= 0).any():
            return None
        y = np.log(xs + b)
        return cls(a=-b, mu=y.mean(), sigma=np.sqrt(np.sum((y - y.mean()) ** 2) / (n - 1)))


class LogP3(Dist):
    """log-Pearson III, parameters from the moments of log x."""
    name, kind = 'LogP3', 'pearson'

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=float)
        if (x <= 0).any():
            return None
        y = np.log(x)
        g = float(stats.skew(y, bias=False))
        if abs(g) < 1e-3:
            return None
        return cls(mu=y.mean(), sigma=y.std(ddof=1), g=g)

    def quantile(self, p):
        return np.exp(self.p['mu'] + self.p['sigma'] * pearson3_K(p, self.p['g']))

    def cdf(self, x):
        x = np.asarray(x, dtype=float)
        with np.errstate(invalid='ignore', divide='ignore'):
            z = (np.log(np.maximum(x, 1e-300)) - self.p['mu']) / self.p['sigma']
        return np.where(x > 0, pearson3_cdf_std(z, self.p['g']), 0.0)

    @staticmethod
    def real_moments(mu, sigma, g):
        """mean, sd, skew of x = exp(y), y ~ Pearson III(mu, sigma, g). None if a moment does not exist."""
        a = 4.0 / g ** 2
        M = []
        for r in (1, 2, 3):
            if g > 0:
                c = 1 - r * sigma * g / 2
                if c <= 0:
                    return None
                M.append(np.exp(r * (mu - 2 * sigma / g)) * c ** (-a))
            else:
                c = 1 + r * sigma * abs(g) / 2
                M.append(np.exp(r * (mu + 2 * sigma / abs(g))) * c ** (-a))
        var = M[1] - M[0] ** 2
        if var <= 0:
            return None
        skew = (M[2] - 3 * M[0] * M[1] + 2 * M[0] ** 3) / var ** 1.5
        return M[0], np.sqrt(var), skew


class LP3Rs(LogP3):
    """log-Pearson III whose real-space mean, variance and skew match the sample (numerical)."""
    name = 'LP3Rs'

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=float)
        if (x <= 0).any():
            return None
        target = np.array([x.mean(), x.std(ddof=1), stats.skew(x, bias=False)])
        start = LogP3.fit(x)
        if start is None:
            return None
        p0 = np.array([start.p['mu'], np.log(start.p['sigma']), start.p['g']])

        def loss(q):
            mu, sigma, g = q[0], np.exp(q[1]), q[2]
            if abs(g) < 1e-4:
                return 1e6
            mom = LogP3.real_moments(mu, sigma, g)
            if mom is None:
                return 1e6
            mom = np.array(mom)
            return float(np.sum(((mom - target) / np.where(np.abs(target) > 1e-9, np.abs(target), 1)) ** 2))

        res = optimize.minimize(loss, p0, method='Nelder-Mead', options={'xatol': 1e-8, 'fatol': 1e-12, 'maxiter': 4000})
        if not np.isfinite(res.fun) or res.fun > 1e-6:
            return None
        mu, sigma, g = res.x[0], np.exp(res.x[1]), res.x[2]
        return cls(mu=mu, sigma=sigma, g=g)


class SqrtEt(Dist):
    """Square-root exponential-type distribution of maximum (Etoh & Murota): F = exp(-a (1 + sqrt(b x)) exp(-sqrt(b x)))."""
    name, kind = 'SqrtEt', 'gumbel'

    @staticmethod
    def _nll(q, x):
        a, b = np.exp(q)
        t = np.sqrt(b * x)
        return float(np.sum(a * (1 + t) * np.exp(-t) - np.log(a * b / 2) + t))

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=float)
        if (x <= 0).any():
            return None
        best = None
        for la in np.linspace(-1, 6, 8):                    # coarse grid, then Nelder-Mead
            for lb in np.linspace(-9, 1, 11):
                v = cls._nll(np.array([la, lb]), x)
                if np.isfinite(v) and (best is None or v < best[0]):
                    best = (v, np.array([la, lb]))
        if best is None:
            return None
        res = optimize.minimize(cls._nll, best[1], args=(x,), method='Nelder-Mead', options={'xatol': 1e-8, 'fatol': 1e-10, 'maxiter': 4000})
        a, b = np.exp(res.x)
        return cls(a=a, b=b) if np.isfinite(a) and np.isfinite(b) else None

    def quantile(self, p):
        p = np.atleast_1d(np.asarray(p, dtype=float))
        a, b = self.p['a'], self.p['b']
        out = np.zeros_like(p)
        for i, pi in enumerate(p):
            rhs = -np.log(pi) / a
            if rhs >= 1:
                out[i] = 0.0
                continue
            t = optimize.brentq(lambda t: (1 + t) * np.exp(-t) - rhs, 0.0, 700.0)
            out[i] = t ** 2 / b
        return out if out.size > 1 else float(out[0])

    def cdf(self, x):
        t = np.sqrt(self.p['b'] * np.maximum(np.asarray(x, dtype=float), 0))
        return np.exp(-self.p['a'] * (1 + t) * np.exp(-t))


DISTRIBUTIONS = [Exp, Gumbel, SqrtEt, GEV, LN2LM, LN3Q, LN3PM, LogP3, LP3Rs, Iwai]


def safe_fit(cls, x):
    try:
        d = cls.fit(x)
        if d is None:
            return None
        q = np.asarray(d.quantile(np.array([0.5, 0.9, 0.99])), dtype=float)
        return d if np.all(np.isfinite(q)) and np.all(np.diff(q) >= 0) else None
    except Exception:
        return None


def slsc(dist, x):
    xs = np.sort(np.asarray(x, dtype=float))
    s_star = dist.reduced(cunnane(len(xs)))
    s = dist.reduced_of_x(xs)
    if not (np.all(np.isfinite(s)) and np.all(np.isfinite(s_star))):
        return np.inf
    den = abs(float(dist.reduced(0.99) - dist.reduced(0.01)))
    return float(np.sqrt(np.mean((s - s_star) ** 2)) / den) if den > 0 else np.inf


def jackknife(cls, x, T=JK_T):
    """Bias-corrected Jackknife estimate and standard error of the T-year quantile."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    full = safe_fit(cls, x)
    if full is None or n < 6:
        return np.nan, np.nan
    theta = float(full.quantile(1 - 1 / T))
    loo = []
    for i in range(n):
        d = safe_fit(cls, np.delete(x, i))
        if d is not None:
            v = float(d.quantile(1 - 1 / T))
            if np.isfinite(v):
                loo.append(v)
    if len(loo) < max(3, n - 2):
        return np.nan, np.nan
    loo = np.array(loo)
    m = len(loo)
    theta_j = n * theta - (n - 1) * loo.mean()
    se = float(np.sqrt((m - 1) / m * np.sum((loo - loo.mean()) ** 2)))
    return theta_j, se


def analyse_series(x, label=''):
    """Fit every candidate; return (scores DataFrame, selected name, selected fitted distribution)."""
    x = np.asarray(x, dtype=float)
    rows, fitted = [], {}
    for cls in DISTRIBUTIONS:
        d = safe_fit(cls, x)
        if d is None:
            rows.append({'distribution': cls.name, 'fitted': False, 'SLSC': np.nan, f'JK{JK_T}_est': np.nan, f'JK{JK_T}_se': np.nan, 'accepted': False})
            continue
        fitted[cls.name] = d
        s = slsc(d, x)
        est, se = jackknife(cls, x)
        rows.append({'distribution': cls.name, 'fitted': True, 'SLSC': round(s, 4) if np.isfinite(s) else np.nan,
                     f'JK{JK_T}_est': round(est, 1) if np.isfinite(est) else np.nan,
                     f'JK{JK_T}_se': round(se, 2) if np.isfinite(se) else np.nan,
                     'accepted': bool(np.isfinite(s) and s <= SLSC_THRESHOLD)})
    df = pd.DataFrame(rows).set_index('distribution')
    ok = df[df['accepted'] & df[f'JK{JK_T}_se'].notna()]
    if not ok.empty:
        best = ok[f'JK{JK_T}_se'].idxmin()
        flag = ''
    else:
        viable = df[df['SLSC'].notna()].sort_values('SLSC')
        best = viable.index[0] if not viable.empty else None
        flag = ' (no distribution reaches SLSC <= %.2f)' % SLSC_THRESHOLD
    df.attrs['selected'] = best
    df.attrs['flag'] = flag
    df.attrs['label'] = label
    return df, best, (fitted.get(best) if best else None)


def rp_values(dist):
    return {f'{T}yr': (round(float(dist.quantile(1 - 1 / T)), 1) if dist is not None else np.nan) for T in ALL_RPS}


RP_TICKS = [2, 5, 10, 20, 50, 100, 200, 400]


def _panel_top(P):
    """Largest value the panel must show: sample maxima, pooled and observation 400-year values."""
    tops = [float(np.max(x)) for x in P['series'].values()]
    if P['pooled'] and P['pooled'][2] is not None:
        tops.append(float(P['pooled'][2].quantile(1 - 1 / RP_TICKS[-1])))
    if P.get('obs') and P['obs'][1] is not None:
        tops.append(float(P['obs'][1].quantile(1 - 1 / RP_TICKS[-1])))
    return max(tops) if tops else 1.0


def draw_probability_panel(ax, P, ylim=None, label=None, legend=False):
    """One scenario on Gumbel probability paper (Cunnane plotting position).

    light-grey points / thin grey lines: annual maxima and selected fit of every GCM;
    black dashed: mean over the models of the fitted quantiles; black solid: fit to the pooled sample;
    red: observation (points and selected fit).
    """
    y_ticks = gumbel_y([1 - 1 / T for T in RP_TICKS])
    n_max = max([len(v) for v in P['series'].values()] + ([len(P['obs'][0])] if P.get('obs') else [1]))
    y_lo = min(gumbel_y(cunnane(n_max)[0]), y_ticks[0]) - 0.2
    y_curve = np.linspace(y_lo, y_ticks[-1] + 0.2, 300)
    p_curve = np.exp(-np.exp(-y_curve))
    curves = []
    for m, x in P['series'].items():
        xs = np.sort(np.asarray(x, dtype=float))
        ax.scatter(gumbel_y(cunnane(len(xs))), xs, s=6, color=COL_GCM_PT, linewidths=0, alpha=0.9, zorder=3)
        d = P['fits'].get(m)
        if d is not None:
            xc = np.asarray(d.quantile(p_curve), dtype=float)
            ax.plot(y_curve, xc, color=COL_GCM, linewidth=0.55, alpha=0.9, zorder=4)
            curves.append(xc)
    handles = [Line2D([], [], color=COL_GCM, lw=0.8, marker='o', markerfacecolor=COL_GCM_PT, markeredgewidth=0,
                      markersize=3, label=f'Individual GCMs (n = {len(P["series"])})')]
    if curves:
        ax.plot(y_curve, np.nanmean(np.vstack(curves), axis=0), color=COL_ENS, linewidth=1.5, linestyle='--', zorder=6)
        handles.append(Line2D([], [], color=COL_ENS, lw=1.5, ls='--', label='Multi-model mean of fitted quantiles'))
    if P['pooled'] and P['pooled'][2] is not None:
        _, xp, d = P['pooled']
        ax.plot(y_curve, np.asarray(d.quantile(p_curve), dtype=float), color=COL_ENS, linewidth=2.0, zorder=6)
        handles.append(Line2D([], [], color=COL_ENS, lw=2.0, label=f'Pooled-sample fit ({d.name}, N = {len(xp)})'))
    if P.get('obs'):
        xo, d, lab = P['obs']
        xs = np.sort(np.asarray(xo, dtype=float))
        ax.scatter(gumbel_y(cunnane(len(xs))), xs, s=20, color=COL_OBS, marker='D', linewidths=0, zorder=7)
        if d is not None:
            ax.plot(y_curve, np.asarray(d.quantile(p_curve), dtype=float), color=COL_OBS, linewidth=1.6, zorder=7)
        handles.append(Line2D([], [], color=COL_OBS, lw=1.6, marker='D', markersize=4, markeredgewidth=0,
                              label=f'{lab} ({d.name if d is not None else "no fit"}, N = {len(xs)})'))
    ax.set_xticks(y_ticks)
    ax.set_xticklabels([str(T) for T in RP_TICKS])
    ax.set_xlim(y_lo, y_ticks[-1] + 0.2)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.text(0.015, 0.96, label or P['sc'], transform=ax.transAxes, va='top', ha='left', fontweight='bold')
    if legend:
        ax.legend(handles=handles, loc='lower right')
    return handles


def probability_figure(panels, base, common_ylim=True):
    """One figure with one panel per scenario, stacked vertically (shared x, common y when common_ylim); PNG + PDF."""
    if not panels:
        return
    top = max(_panel_top(P) for P in panels)
    ylim = (0, top * 1.08) if common_ylim else (0, _panel_top(panels[0]) * 1.08)
    n = len(panels)
    fig, axes = plt.subplots(n, 1, figsize=((6.0, 3.0 * n + 0.5) if n > 1 else (5.5, 4.8)), sharex=True, squeeze=False)
    for i, (ax, P) in enumerate(zip(axes[:, 0], panels)):
        draw_probability_panel(ax, P, ylim=ylim, label=(f'({chr(97 + i)}) {P["sc"]}' if n > 1 else P['sc']), legend=(i == 0))
    axes[-1, 0].set_xlabel('Return period (years)')
    fig.supylabel('Annual maximum daily basin-average rainfall (mm)')
    fig.tight_layout(h_pad=0.5)
    save_figure(fig, base)
    plt.show()


In [ ]:
# ============================================================
# Section 3b: Per-model fitting, pooled sample, observation; probability plots
# ============================================================
by_model_rows, score_rows = [], []
fits = {}            # (scenario, model) -> fitted distribution
series = {}          # (scenario, model) -> annual maxima
pooled_fits = {}     # scenario -> (name, x, dist)
obs_result = None
panels = []          # inputs of the probability-paper panels, one per scenario

for sc, ann in annual_max.items():
    models = [c for c in ann.columns if c != 'Ensemble']
    fit_by_model = {}
    for m in models:
        x = ann[m].dropna().values
        if len(x) < 6:
            print(f'  ⚠ {sc}/{m}: only {len(x)} years, skipped')
            continue
        scores, best, d = analyse_series(x, f'{sc}/{m}')
        series[(sc, m)] = x
        fits[(sc, m)] = d
        fit_by_model[m] = d
        for dist_name, r in scores.iterrows():
            score_rows.append({'scenario': sc, 'series': m, 'distribution': dist_name, **r.to_dict(), 'selected': dist_name == best})
        by_model_rows.append({'scenario': sc, 'series': m, 'N': len(x), 'selected': best,
                              'SLSC': scores.loc[best, 'SLSC'] if best else np.nan,
                              f'JK{JK_T}_se': scores.loc[best, f'JK{JK_T}_se'] if best else np.nan,
                              'note': scores.attrs['flag'], **rp_values(d)})
    if not fit_by_model:
        n_years = {m: int(ann[m].dropna().shape[0]) for m in models}
        print(f'  ⚠ {sc}: no model has at least 6 years of annual maxima (years per model: {n_years}) — '
              f'the {sc}_corr files probably come from a partial download (e.g. QUICK_TEST); scenario skipped')
        continue
    # pooled sample: all models' annual maxima of this scenario
    xp = np.concatenate([series[(sc, m)] for m in models if (sc, m) in series])
    scores, best, d = analyse_series(xp, f'{sc}/pooled')
    pooled_fits[sc] = ('pooled', xp, d)
    for dist_name, r in scores.iterrows():
        score_rows.append({'scenario': sc, 'series': 'pooled', 'distribution': dist_name, **r.to_dict(), 'selected': dist_name == best})
    by_model_rows.append({'scenario': sc, 'series': 'pooled', 'N': len(xp), 'selected': best,
                          'SLSC': scores.loc[best, 'SLSC'] if best else np.nan,
                          f'JK{JK_T}_se': scores.loc[best, f'JK{JK_T}_se'] if best else np.nan,
                          'note': scores.attrs['flag'], **rp_values(d)})
    sel_txt = ', '.join(f'{m}: {fit_by_model[m].name if fit_by_model.get(m) else "-"}' for m in fit_by_model)
    print(f'{sc}: selected per model -> {sel_txt}; pooled -> {best}{scores.attrs["flag"]}')

    obs_arg = None
    if sc == 'historical' and obs_annual_max:
        xo = obs_annual_max['Observation'].iloc[:, 0].dropna().values
        if len(xo) >= 6:
            scores_o, best_o, d_o = analyse_series(xo, 'Observation')
            obs_result = (xo, d_o, scores_o, best_o)
            for dist_name, r in scores_o.iterrows():
                score_rows.append({'scenario': 'Observation', 'series': obs_source_label, 'distribution': dist_name, **r.to_dict(), 'selected': dist_name == best_o})
            by_model_rows.append({'scenario': 'Observation', 'series': obs_source_label, 'N': len(xo), 'selected': best_o,
                                  'SLSC': scores_o.loc[best_o, 'SLSC'] if best_o else np.nan,
                                  f'JK{JK_T}_se': scores_o.loc[best_o, f'JK{JK_T}_se'] if best_o else np.nan,
                                  'note': scores_o.attrs['flag'], **rp_values(d_o)})
            obs_arg = (xo, d_o, f'Observation {obs_source_label}')
            print(f'Observation ({obs_source_label}): selected -> {best_o}{scores_o.attrs["flag"]}')
    panels.append({'sc': sc, 'series': {m: series[(sc, m)] for m in models if (sc, m) in series},
                   'fits': fit_by_model, 'pooled': pooled_fits[sc], 'obs': obs_arg})
    probability_figure([panels[-1]], f'{ANALYSIS_DIR}/prob_plot_{sc}', common_ylim=False)

if len(panels) > 1:
    print('\nCombined figure (all scenarios, common y-axis):')
    probability_figure(panels, f'{ANALYSIS_DIR}/prob_plot_all')

by_model_df = pd.DataFrame(by_model_rows)
scores_df = pd.DataFrame(score_rows)
by_model_df.to_csv(f'{ANALYSIS_DIR}/return_periods_by_model.csv', index=False)
scores_df.to_csv(f'{ANALYSIS_DIR}/fit_scores.csv', index=False)
print(f'\nSaved: {ANALYSIS_DIR}/return_periods_by_model.csv, {ANALYSIS_DIR}/fit_scores.csv')


In [ ]:
# ============================================================
# Section 3c: Summary — multi-model statistics of the return-period rainfall
# ============================================================
rp_cols = [f'{T}yr' for T in ALL_RPS]
summary_rows = []
for sc in annual_max:
    per_model = by_model_df[(by_model_df['scenario'] == sc) & (by_model_df['series'] != 'pooled')]
    pooled_sel = by_model_df[(by_model_df['scenario'] == sc) & (by_model_df['series'] == 'pooled')]
    if per_model.empty or pooled_sel.empty:
        print(f'  ⚠ {sc}: no frequency-analysis result (see the messages above) — left out of the summary')
        continue
    n_acc = int((per_model['note'] == '').sum())
    for stat, fn in (('mean over models', 'mean'), ('median over models', 'median'), ('min over models', 'min'), ('max over models', 'max')):
        summary_rows.append({'scenario': sc, 'statistic': stat, 'models': len(per_model), 'models with SLSC<=0.04': n_acc,
                             **per_model[rp_cols].agg(fn).round(1).to_dict()})
    pooled_row = pooled_sel.iloc[0]
    summary_rows.append({'scenario': sc, 'statistic': f'pooled fit ({pooled_row["selected"]}, N={int(pooled_row["N"])})',
                         'models': len(per_model), 'models with SLSC<=0.04': n_acc, **pooled_row[rp_cols].to_dict()})
if obs_result is not None:
    orow = by_model_df[by_model_df['scenario'] == 'Observation'].iloc[0]
    summary_rows.append({'scenario': 'Observation', 'statistic': f'{obs_source_label} ({orow["selected"]}, N={int(orow["N"])})',
                         'models': 1, 'models with SLSC<=0.04': int(orow['note'] == ''), **orow[rp_cols].to_dict()})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f'{ANALYSIS_DIR}/return_periods_summary.csv', index=False)
print(f'Saved: {ANALYSIS_DIR}/return_periods_summary.csv\n')
print('Return-period rainfall (mm/day, basin average) — per scenario:')
print(summary_df.to_string(index=False))
print('\nPer model (selected distribution, SLSC, Jackknife SE of the %d-year value):' % JK_T)
print(by_model_df[['scenario', 'series', 'N', 'selected', 'SLSC', f'JK{JK_T}_se', 'note'] + ['10yr', '50yr', '100yr', '200yr']].to_string(index=False))

excel_path = f'{ANALYSIS_DIR}/return_periods.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as xw:
    summary_df.to_excel(xw, sheet_name='summary', index=False)
    by_model_df.to_excel(xw, sheet_name='by_model', index=False)
    scores_df.to_excel(xw, sheet_name='fit_scores', index=False)
    mk_df.to_excel(xw, sheet_name='mann_kendall', index=False)
print(f'\nSaved: {excel_path}')
